# Inference Lab: Pricing Under Uncertainty


**Run this notebook in Google Colab.** The `demand` module is compiled for Linux and Python 3.13 only, so it will not import on a Mac or Windows machine.

The goal of this lab is to use what you've learned in this class to find the optimal price for something. You can't know for sure what the underlying curve of revenue vs price looks like. All you know is what you would know in the real world, information you gain by sampling-- you can market test any price using the demand function below. When you set the price, the market test runs for a certain amount of time, and then reports responses-- a list of 1s (bought) and 0s (didn't buy) for that price point. The total number in the sample, $n$, is therefore a random variable, given by `len(responses)`. The number of consumers who buy the product is `responses.sum().` Every time you run a market test, it takes time (literally, it will take a while to run, so don't do this lab last minute!).

In [1]:
!git clone -q https://github.com/smgroves/demand_lab.git

%cd /content/demand_lab

from demand import demand

responses = demand(price=14.0)

print(len(responses), responses.sum())

/content/demand_lab
206 165


In [2]:
import numpy as np

## 1. Compare two prices

Run one market test with price = 30. Then test price = 25. Which price earns more revenue per potential customer?

1. For each price, estimate the proportion $\hat{p}$ of customers who buy, and the revenue per customer $\hat{r}$.
2. Estimate the standard error of each $\hat{r}$ by bootstrapping.
3. Compute a 95% confidence interval for each $\hat{r}$.
4. Compute a 95% confidence interval for the difference $\hat{r}_{25} - \hat{r}_{30}$.
   
5. Interpret the estimates, SEs, and intervals. Report which price earns more profit, and how confident you are.



In [3]:
resp_30 = demand(price=30.0)
resp_25 = demand(price=25.0)

In [4]:
R = 5_000
np.random.seed(1)

n_30, n_25 = len(resp_30), len(resp_25)
p_30, p_25 = resp_30.mean(), resp_25.mean()
r_30, r_25 = 30 * p_30, 25 * p_25
reps_30 = []
reps_25 = []
for r in range(R):
    idx_30 = np.random.randint(0, n_30, n_30)
    idx_25 = np.random.randint(0, n_25, n_25)
    reps_30.append(30 * resp_30[idx_30].mean())
    reps_25.append(25 * resp_25[idx_25].mean())
reps_30 = np.array(reps_30)
reps_25 = np.array(reps_25)
reps_diff = reps_25 - reps_30

print('30: n =', n_30, ', p hat =', round(p_30, 3), ', r hat =', round(r_30, 3))
print('25: n =', n_25, ', p hat =', round(p_25, 3), ', r hat =', round(r_25, 3))
print('SE 30:', round(reps_30.std(), 3))
print('SE 25:', round(reps_25.std(), 3))
print('95% CI 30:', np.quantile(reps_30, [0.025, 0.975]).round(3))
print('95% Ci 25:', np.quantile(reps_25, [0.025, 0.975]).round(3))
print('r_25 - r_30 =', round(r_25 - r_30, 3))
print('95% CI diff:', np.quantile(reps_diff, [0.025, 0.975]).round(3))

30: n = 154 , p hat = 0.442 , r hat = 13.247
25: n = 220 , p hat = 0.414 , r hat = 10.341
SE 30: 1.195
SE 25: 0.816
95% CI 30: [10.909 15.584]
95% Ci 25: [ 8.75  11.932]
r_25 - r_30 = -2.906
95% CI diff: [-5.747 -0.162]


### 5

About 44\% of customers bought at \$30, and revenue per customer is around \$13.25 At $25, around 41\% of customers bought,  for around \$10.34 per customer.The 95\% CI for the difference (25 minus 30) is [-5.75, -0.16]. Since this does not include 0, \$30 earns more per customer than \$25. Since the upper end is very close to 0, the difference could be fairly small, but we are fairly confident.

## 2. The cost of sampling

7. Run a second market test at each price. Pool the two price = 30 samples (`np.concatenate`), and pool the two price = 25 samples.
8. Recompute the SEs, the intervals, and the interval for the difference. By roughly what factor did the SE shrink? Explain why, using the SE formula.
   
9.  **Was the extra sampling worth it?** Every market test takes time, and while you're testing you aren't selling at your best price, so each test costs the company revenue. Did the first round already settle the question (its difference interval excluded 0)? If so, the second round cost time and bought nothing but a narrower interval. If not, did the second round settle it?


In [5]:
resp2_30 = demand(price=30.0)
resp2_25 = demand(price=25.0)

In [6]:
R = 5_000
np.random.seed(1)

pool_30 = np.concatenate([resp_30, resp2_30])
pool_25 = np.concatenate([resp_25, resp2_25])
pn_30, pn_25 = len(pool_30), len(pool_25)
pp_30, pp_25 = pool_30.mean(), pool_25.mean()
pr_30, pr_25 = 30 * pp_30, 25 * pp_25
preps_30 = []
preps_25 = []
for r in range(R):
    i30 = np.random.randint(0, pn_30, pn_30)
    i25 = np.random.randint(0, pn_25, pn_25)
    preps_30.append(30 * pool_30[i30].mean())
    preps_25.append(25 * pool_25[i25].mean())
preps_30 = np.array(preps_30)
preps_25 = np.array(preps_25)
preps_diff = preps_25 - preps_30

print('30: n =', pn_30, ', p hat =', round(pp_30, 3), ', r hat =', round(pr_30, 3))
print('25: n =', pn_25, ', p hat =', round(pp_25, 3), ', r hat =', round(pr_25, 3))
print('SE 30:', round(preps_30.std(), 3))
print('SE 25:', round(preps_25.std(), 3))
print('95% CI 30:', np.quantile(preps_30, [0.025, 0.975]).round(3))
print('95% CI 25:', np.quantile(preps_25, [0.025, 0.975]).round(3))
print('r_25 - r_30 =', round(pr_25 - pr_30, 3))
print('95% CI diff:', np.quantile(preps_diff, [0.025, 0.975]).round(3))
print('SE ratio 30:', round(preps_30.std() / reps_30.std(), 3))
print('SE ratio 25:', round(preps_25.std() / reps_25.std(), 3))

30: n = 322 , p hat = 0.388 , r hat = 11.646
25: n = 394 , p hat = 0.388 , r hat = 9.708
SE 30: 0.815
SE 25: 0.619
95% CI 30: [10.062 13.23 ]
95% CI 25: [ 8.503 10.914]
r_25 - r_30 = -1.938
95% CI diff: [-3.873  0.06 ]
SE ratio 30: 0.683
SE ratio 25: 0.759


### 8

The SE shrank to around 0.68 of its original size at \$30, and about 0.76 at \$25, so it roughly shrinks by around 0.7. The SE of $\hat{r}$ is $P \cdot \sqrt{(\hat{p}(1-\hat{p})/n)}$, and the pooling is around 2x from before. Since $n$ is under the square root, doubling it multiplies the SE by $\frac{1}{\sqrt{2}} \approx 0.71$.

### 9

The earlier interval did not include 0, so it showed \$30 earning more. With this data, the interval now includes 0, so the second round didn't settle the question. It shows that the first results may be a fluke, and the gap between \$25 and \$30 is likely small. It doesn't determine anything for us, but it does keep us from being overconfident.

## 3. Finding the optimal price

Now find the price that maximizes revenue per customer. Search prices between about \$5 and \$100.

Every market test takes time, and time spent testing is revenue lost: while you test, you aren't selling at your best price. Your confidence intervals tell you whether another test is worth running. **Use as few market tests as you can (<25 would be good).** Your grade will be based on, in this order, whether you:
- accurately used the information you have by accurately interpreting confidence intervals
- found a good price, close to the optimal price
- and used fewer tests.

If this sounds fun to you, run with it! If you are still very confused about what to do, read the hints below.

<details><Summary>Hints</summary>
<b>Hint 1:</b> There are a few ways you could search:

- Brute force: test every price on a fine grid. This takes a market test at every price, including prices that are obviously bad. This would be a bad choice.
- Hill climbing: compare the CI for two prices, move toward the better one, and repeat. For example, start from the 25/30 comparison above; based on which price is better, choose another price \$10 away in that direction (15 if 25 was better; 40 if 30 was better). Shrink the step size as you get close to a good price. If you can't detect a difference between two prices, either shrink the step size or increase the sample size (i.e. run another market test at the same price).
- other strategies that I won't mention here.
  
<b>Hint 2:</b> You may want to try starting your tests with a few different prices across the range; there may be local maxima you will get stuck in if you just hill climb towards a single maximum.

<b>Hint 3:</b> You can make up your own strategy or try a mix of strategies. However, keep in mind that your grade will reflect how well you ran A/B testing and interpreting confidence intervals, more than how complex your strategy is. Similarly, a lucky guess with a brute force strategy would not be a good reproducible strategy.

<b>Hint 4:</b>
Turn your code from part 1 into a function, so you can easily input responses, run bootstrapping, calculate SE, and calculate confidence intervals. Then make another function where you give it two confidence intervals and make a decision based on them: sample more at the same test, or move in one direction with some step size.
   </details>

**Before you run any tests,** write down your plan: which prices you'll test first, how you'll decide where to go next, and what your intervals need to show before you stop. Then follow it, and note anywhere you changed course and why.

For any method, stop when your intervals show that another test is unlikely to change your answer by enough to matter.

**Deliverable:** A ~2-3 paragraph write up of the strategy you chose, your final recommended price, a 95% CI for $r$ at that price, the number of market tests you ran, and one or two sentences on how your intervals told you when to stop.

I will do one test at each of \$45, \$60, \$75, and \$90 to check for any local maximums. Whichever has the highest $\hat{r}$, I'll compare with neighbors using the 95\% CI for difference. If a neighbor excludes 0, we move in that direction, and if it includes 0,  we either test halfway in between or run another test. I will stop when the best price isn't told apart from its nearby neighbors and the estimated difference is small

In [7]:
np.random.seed(1)
tests = 5
data = {14.0: responses, 25.0: pool_25, 30.0: pool_30}

def test(P):
    global tests
    resp = demand(price=P)
    if P in data:
        data[P] = np.concatenate([data[P], resp])
    else:
        data[P] = resp
    tests += 1
    print(P, ', tests:', tests)

def boot(P, R=5_000):
    resp = data[P]
    n = len(resp)
    reps = []
    for r in range(R):
        idx = np.random.randint(0, n, n)
        reps.append(P * resp[idx].mean())
    return np.array(reps)

def summary():
    for P in sorted(data):
        reps = boot(P)
        print(P, ': n =', len(data[P]), ', r hat =', round(P * data[P].mean(), 3), ', 95% CI', np.quantile(reps, [0.025, 0.975]).round(3))

def compare(A, B):
    diff = boot(A) - boot(B)
    print('r_' + str(A), '- r_' + str(B), '=', round(A * data[A].mean() - B * data[B].mean(), 3), ', 95% CI', np.quantile(diff, [0.025, 0.975]).round(3))

In [8]:
test(45.0)
test(60.0)
test(75.0)
test(90.0)
summary()

45.0 , tests: 6
60.0 , tests: 7
75.0 , tests: 8
90.0 , tests: 9
14.0 : n = 206 , r hat = 11.214 , 95% CI [10.466 11.961]
25.0 : n = 394 , r hat = 9.708 , 95% CI [ 8.503 10.914]
30.0 : n = 322 , r hat = 11.646 , 95% CI [10.062 13.23 ]
45.0 : n = 157 , r hat = 17.484 , 95% CI [14.045 20.924]
60.0 : n = 138 , r hat = 16.522 , 95% CI [12.174 20.87 ]
75.0 : n = 145 , r hat = 20.172 , 95% CI [14.483 25.862]
90.0 : n = 211 , r hat = 17.915 , 95% CI [13.223 23.033]


In [9]:
compare(45.0, 30.0)
compare(75.0, 45.0)
compare(75.0, 60.0)
compare(75.0, 90.0)

r_45.0 - r_30.0 = 5.838 , 95% CI [2.047 9.665]
r_75.0 - r_45.0 = 2.688 , 95% CI [-3.632  9.302]
r_75.0 - r_60.0 = 3.651 , 95% CI [-3.261 11.102]
r_75.0 - r_90.0 = 2.258 , 95% CI [-4.866  9.291]


In [10]:
test(45.0)
test(75.0)
test(90.0)
summary()

45.0 , tests: 10
75.0 , tests: 11
90.0 , tests: 12
14.0 : n = 206 , r hat = 11.214 , 95% CI [10.466 11.961]
25.0 : n = 394 , r hat = 9.708 , 95% CI [ 8.503 10.977]
30.0 : n = 322 , r hat = 11.646 , 95% CI [10.062 13.323]
45.0 : n = 371 , r hat = 17.466 , 95% CI [15.283 19.65 ]
60.0 : n = 138 , r hat = 16.522 , 95% CI [12.174 20.87 ]
75.0 : n = 301 , r hat = 21.429 , 95% CI [17.691 25.166]
90.0 : n = 356 , r hat = 18.961 , 95% CI [15.169 22.753]


In [11]:
compare(75.0, 45.0)
compare(75.0, 60.0)
compare(75.0, 90.0)

r_75.0 - r_45.0 = 3.962 , 95% CI [-0.388  8.408]
r_75.0 - r_60.0 = 4.907 , 95% CI [-0.941 10.569]
r_75.0 - r_90.0 = 2.468 , 95% CI [-2.846  7.771]


In [12]:
test(67.5)
test(82.5)
summary()
compare(75.0, 67.5)
compare(75.0, 82.5)

67.5 , tests: 13
82.5 , tests: 14
14.0 : n = 206 , r hat = 11.214 , 95% CI [10.466 11.961]
25.0 : n = 394 , r hat = 9.708 , 95% CI [ 8.503 10.914]
30.0 : n = 322 , r hat = 11.646 , 95% CI [10.062 13.23 ]
45.0 : n = 371 , r hat = 17.466 , 95% CI [15.162 19.65 ]
60.0 : n = 138 , r hat = 16.522 , 95% CI [12.174 21.304]
67.5 : n = 165 , r hat = 18.0 , 95% CI [13.5   22.909]
75.0 : n = 301 , r hat = 21.429 , 95% CI [17.691 25.415]
82.5 : n = 179 , r hat = 23.045 , 95% CI [17.514 28.575]
90.0 : n = 356 , r hat = 18.961 , 95% CI [15.421 23.006]
r_75.0 - r_67.5 = 3.429 , 95% CI [-2.585  9.424]
r_75.0 - r_82.5 = -1.616 , 95% CI [-8.294  5.024]


In [13]:
test(82.5)
summary()
compare(82.5, 75.0)
compare(82.5, 90.0)

82.5 , tests: 15
14.0 : n = 206 , r hat = 11.214 , 95% CI [10.466 11.961]
25.0 : n = 394 , r hat = 9.708 , 95% CI [ 8.503 10.914]
30.0 : n = 322 , r hat = 11.646 , 95% CI [10.062 13.323]
45.0 : n = 371 , r hat = 17.466 , 95% CI [15.283 19.771]
60.0 : n = 138 , r hat = 16.522 , 95% CI [12.174 21.304]
67.5 : n = 165 , r hat = 18.0 , 95% CI [13.5   22.909]
75.0 : n = 301 , r hat = 21.429 , 95% CI [17.691 25.415]
82.5 : n = 364 , r hat = 21.758 , 95% CI [18.132 25.611]
90.0 : n = 356 , r hat = 18.961 , 95% CI [15.169 23.006]
r_82.5 - r_75.0 = 0.33 , 95% CI [-4.906  5.881]
r_82.5 - r_90.0 = 2.798 , 95% CI [-2.476  8.351]


I started by testing \$45, \$60, \$75, \$90 once each, since we already have some lower values. The low prices are clearly worse, as the 95\% CI for 45-30 was [2.05, 9.67], which doesn't include 0.

Since I couldn't tell the high prices apart based on their similar estimates, I ran a second test at \$45, \$75, \$90, where \$75 had the highest peak. I halved by step size and tested \$67.5 and \$82.5, where \$82.5 was better. I gave it a second test to compare it with \$75 and \$90.

My recommendation is \$82.5, with an estimated \$21.76 revenue per customer and a 95\% CI of [18.13, 25.61]. I used 15 market tests, and I stopped since the 95\% CI for 82.5 - 75 was [-4.91, 5.88] (includes 0) and the estimated difference was only 0.33.